# Chapter 11 — Q-Learning: Live Example on FrozenLake

**Session 4 | Chapter 2 | Duration: ~10 min**

We implement Q-Learning from scratch and train it on the FrozenLake environment.

> **Requirements:** `pip install gymnasium`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import gymnasium as gym
    GYM_AVAILABLE = True
    print('Gymnasium available!')
except ImportError:
    GYM_AVAILABLE = False
    print('Gymnasium not installed. Run: pip install gymnasium')
    print('Falling back to built-in grid world...')

sns.set_theme(style='whitegrid')
np.random.seed(42)

## 1. FrozenLake Environment

```
SFFF       S=Start (0)
FHFH       F=Frozen (safe)
FFFH       H=Hole (game over)
HFFG       G=Goal (+1 reward)
```
- 16 states (4×4 grid)
- 4 actions: Left=0, Down=1, Right=2, Up=3
- Slippery ice: sometimes moves sideways!

In [ ]:
if GYM_AVAILABLE:
    # Use actual Gymnasium FrozenLake
    env = gym.make('FrozenLake-v1', is_slippery=False)  # non-slippery for easier learning
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    print(f'FrozenLake: {n_states} states, {n_actions} actions')
    
    def env_reset_gym():
        state, _ = env.reset()
        return state
    
    def env_step_gym(action):
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        return obs, reward, done
    
    reset_fn = env_reset_gym
    step_fn  = env_step_gym
else:
    # Fallback: use our own grid world from Ch10
    GRID = np.array([[3,0,0,0],[0,1,0,1],[0,0,0,1],[1,0,0,2]])
    ROWS, COLS = 4, 4
    n_states, n_actions = 16, 4
    MOVE_DR = {0:(0,-1),1:(1,0),2:(0,1),3:(-1,0)}
    REWARDS = {0:-0.01,1:-1.0,2:1.0,3:-0.01}
    
    _state = [0]
    def reset_fn():
        _state[0] = 0
        return 0
    def step_fn(action):
        r, c = _state[0]//COLS, _state[0]%COLS
        dr, dc = MOVE_DR[action]
        nr, nc = max(0,min(ROWS-1,r+dr)), max(0,min(COLS-1,c+dc))
        ns = nr*COLS+nc
        ct = GRID[nr,nc]
        reward = REWARDS[ct]
        done = ct in [1,2]
        _state[0] = ns
        return ns, reward, done
    
    print('Using built-in grid world!')

print(f'States: {n_states}, Actions: {n_actions}')

## 2. Q-Learning Implementation

In [ ]:
# ──────────────────────────────────────────────
# Q-Learning Hyperparameters
# ──────────────────────────────────────────────
ALPHA   = 0.8    # learning rate
GAMMA   = 0.95   # discount factor
EPSILON = 1.0    # initial exploration rate
EPSILON_DECAY = 0.995
EPSILON_MIN   = 0.01
N_EPISODES    = 5000
MAX_STEPS     = 200

# Initialize Q-table with zeros
Q = np.zeros((n_states, n_actions))

# ──────────────────────────────────────────────
# Training Loop
# ──────────────────────────────────────────────
episode_rewards = []
success_history = []
epsilon = EPSILON

for episode in range(N_EPISODES):
    state = reset_fn()
    total_reward = 0
    success = False
    
    for step in range(MAX_STEPS):
        # ε-greedy action selection
        if np.random.random() < epsilon:
            action = np.random.randint(n_actions)   # explore
        else:
            action = np.argmax(Q[state])             # exploit
        
        # Take action
        next_state, reward, done = step_fn(action)
        total_reward += reward
        
        # Bellman update: Q(s,a) ← Q(s,a) + α·[r + γ·max Q(s',a') - Q(s,a)]
        td_target = reward + GAMMA * np.max(Q[next_state]) * (1 - done)
        td_error  = td_target - Q[state, action]
        Q[state, action] += ALPHA * td_error
        
        state = next_state
        
        if done:
            success = (reward > 0)  # reached goal
            break
    
    episode_rewards.append(total_reward)
    success_history.append(float(success))
    
    # Decay epsilon
    epsilon = max(epsilon * EPSILON_DECAY, EPSILON_MIN)

print('Training complete!')
print(f'Final epsilon: {epsilon:.4f}')
print(f'Success rate (last 500 eps): {np.mean(success_history[-500:]):.1%}')

## 3. Visualize Training Progress

In [ ]:
# Rolling success rate
window = 200
rolling_success = [np.mean(success_history[max(0,i-window):i+1]) for i in range(len(success_history))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Success rate over episodes
axes[0].plot(rolling_success, color='#2ecc71', linewidth=2)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel(f'Success Rate (rolling {window})')
axes[0].set_title('Q-Learning: Agent Gets Better Over Time!', fontsize=13)
axes[0].axhline(np.mean(success_history[-500:]), color='red', linestyle='--', alpha=0.6,
                 label=f'Final: {np.mean(success_history[-500:]):.1%}')
axes[0].legend()

# Reward over episodes
rolling_reward = [np.mean(episode_rewards[max(0,i-window):i+1]) for i in range(len(episode_rewards))]
axes[1].plot(rolling_reward, color='#3498db', linewidth=2)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel(f'Average Reward (rolling {window})')
axes[1].set_title('Average Reward per Episode', fontsize=13)

plt.suptitle('Q-Learning Training Progress', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Visualize the Learned Q-Table

In [ ]:
action_names = ['←', '↓', '→', '↑']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, (ax, name) in enumerate(zip(axes, action_names)):
    q_grid = Q[:, i].reshape(4, 4)
    sns.heatmap(q_grid, annot=True, fmt='.2f', cmap='RdYlGn',
                ax=ax, cbar=False, linewidths=0.5, linecolor='gray')
    ax.set_title(f'Q-values for {name}', fontsize=12)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

plt.suptitle('Learned Q-Table (one heatmap per action)\nHigher value = better to take this action in this cell',
             fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

# Best action per state
best_actions = np.argmax(Q, axis=1)
policy_grid = np.array([action_names[a] for a in best_actions]).reshape(4, 4)

fig, ax = plt.subplots(figsize=(5, 4))
for r in range(4):
    for c in range(4):
        ax.text(c + 0.5, 3.5 - r, policy_grid[r, c], ha='center', va='center',
                fontsize=20, fontweight='bold')
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.grid(True, linewidth=1.5)
ax.set_title('Learned Policy (best action per cell)', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

The agent started with **zero knowledge** (Q-table all zeros) and learned to navigate  
the frozen lake through thousands of trial-and-error episodes.

Key observations:
- Success rate starts near 0%, climbs to high values as Q-values converge
- The Q-table shows higher values near the goal path
- The policy grid shows the arrow pointing in the right direction for each cell

**Chapter 12: Capstone** — Put your full ML knowledge to work on a real-world problem!